### Bayes Model Selection

The notebook implement related stuffs mentioned in https://www.overleaf.com/project/6a7592f575d67477ad61c188

For this notebook, the author will choose StatLog Heart as the dataset for analyzing due to its simplicity.  

### Data Preprocessing

In [58]:
from ucimlrepo import fetch_ucirepo

statlog_heart = fetch_ucirepo(id=145)
X = statlog_heart.data.features
y = statlog_heart.data.targets 
X.describe()

,age,sex,chest-pain,rest-bp,serum-chol,fasting-blood-sugar,electrocardiographic,max-heart-rate,angina,oldpeak,slope,major-vessels,thal
count,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.00000,270.000000,270.000000,270.000000
mean,54.433333,0.677778,3.174074,131.344444,249.659259,0.148148,1.022222,149.677778,0.329630,1.05000,1.585185,0.670370,4.696296
std,9.109067,0.468195,0.950090,17.861608,51.686237,0.355906,0.997891,23.165717,0.470952,1.14521,0.614390,0.943896,1.940659
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.00000,1.000000,0.000000,3.000000
25%,48.000000,0.000000,3.000000,120.000000,213.000000,0.000000,0.000000,133.000000,0.000000,0.00000,1.000000,0.000000,3.000000
50%,55.000000,1.000000,3.000000,130.000000,245.000000,0.000000,2.000000,153.500000,0.000000,0.80000,2.000000,0.000000,3.000000
75%,61.000000,1.000000,4.000000,140.000000,280.000000,0.000000,2.000000,166.000000,1.000000,1.60000,2.000000,1.000000,7.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.20000,3.000000,3.000000,7.000000


Some basic stats on the features 

In [59]:
X.head()

,age,sex,chest-pain,rest-bp,serum-chol,fasting-blood-sugar,electrocardiographic,max-heart-rate,angina,oldpeak,slope,major-vessels,thal
0,70.0,1.0,4.0,130.0,322.0,0.0,2.0,109.0,0.0,2.4,2.0,3.0,3.0
1,67.0,0.0,3.0,115.0,564.0,0.0,2.0,160.0,0.0,1.6,2.0,0.0,7.0
2,57.0,1.0,2.0,124.0,261.0,0.0,0.0,141.0,0.0,0.3,1.0,0.0,7.0
3,64.0,1.0,4.0,128.0,263.0,0.0,0.0,105.0,1.0,0.2,2.0,1.0,7.0
4,74.0,0.0,2.0,120.0,269.0,0.0,2.0,121.0,1.0,0.2,1.0,1.0,3.0


In [60]:
y.head()

,heart-disease
0,2
1,1
2,2
3,1
4,1


In [61]:
print(f"no columns: {len(X.columns)}")
print(X.columns.tolist())

no columns: 13
['age', 'sex', 'chest-pain', 'rest-bp', 'serum-chol', 'fasting-blood-sugar', 'electrocardiographic', 'max-heart-rate', 'angina', 'oldpeak', 'slope', 'major-vessels', 'thal']


Now, we will conduct data clean up with:

- Keep 8 columns for predictor
- Define binary target
- Standardize every non-intercept column

In [62]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
chosen_columns = [0, 1, 3, 4, 5, 7, 8, 9]
scaled_columns = [0, 3, 4, 7, 9]
X.iloc[:, scaled_columns] = scaler.fit_transform(
    X.iloc[:, scaled_columns]
)

chosen_X = X.iloc[: ,chosen_columns]
chosen_y = y.copy()
chosen_y["heart-disease"] = chosen_y["heart-disease"].replace(
    {
        2: 1,
        1: 0, 
    }
)

In [63]:
chosen_X.head()

,age,sex,rest-bp,serum-chol,fasting-blood-sugar,max-heart-rate,angina,oldpeak
0,1.712094,1.0,-0.075410,1.402212,0.0,-1.759208,0.0,1.181012
1,1.382140,0.0,-0.916759,6.093004,0.0,0.446409,0.0,0.481153
2,0.282294,1.0,-0.411950,0.219823,0.0,-0.375291,0.0,-0.656118
3,1.052186,1.0,-0.187590,0.258589,0.0,-1.932198,1.0,-0.743600
4,2.152032,0.0,-0.636310,0.374890,0.0,-1.240239,1.0,-0.743600


In [64]:
chosen_y

,heart-disease
0,1
1,0
2,1
3,0
4,0
...,...
265,0
266,0
267,0
268,0


Okay, now let's proceed to different stages of the main project

In [65]:
from libs import MCMC, AcceptanceTracker, SMC 

### Stage 1: Bayesian Logistic without Variable Selection

- Fit Bayesian Logistic Regression for all selected predictor
- SMC to sample from posterior
- Report Posterior Means, Credible Interval, Posterior Predictive Perf
- SMC est of Normalizing Constant.

The posterior is follow t

In [ ]:
from collections import namedtuple
from jax.scipy.stats import norm
from jax import numpy as jnp

LogisticRegParams = namedtuple("LogisticRegParams", ["alpha", "betas"])


def _to_named_params(params):
    """Accept LogisticRegParams or flat/(d,1) array and return LogisticRegParams."""
    if isinstance(params, LogisticRegParams):
        alpha = jnp.asarray(params.alpha).reshape(())
        betas = jnp.ravel(jnp.asarray(params.betas))
        return LogisticRegParams(alpha=alpha, betas=betas)

    flat = jnp.ravel(jnp.asarray(params))
    return LogisticRegParams(alpha=flat[0], betas=flat[1:])


def prior_log_pdf(params):
    p = _to_named_params(params)
    log_alpha = norm.logpdf(p.alpha, loc=0.0, scale=3.0)
    log_betas = jnp.sum(norm.logpdf(p.betas, loc=0.0, scale=1.5))
    return log_alpha + log_betas


def likelihood_log_pdf(params, X_observed, y_observed):
    p = _to_named_params(params)
    X_observed = jnp.atleast_2d(jnp.asarray(X_observed))
    y_observed = jnp.ravel(jnp.asarray(y_observed))

    # shape-safe linear predictor (works for 1D/2D beta inputs)
    eta = p.alpha + jnp.matmul(X_observed, p.betas[:, None]).squeeze(-1)
    return jnp.sum(y_observed * eta - jnp.logaddexp(0.0, eta))


def posterior_log_pdf(params, X_observed, y_observed):
    return prior_log_pdf(params) + likelihood_log_pdf(params, X_observed, y_observed)


In [67]:
import jax.numpy as jnp
from jax import random


def proposed_fn(params, key):
    """Random-walk proposal preserving LogisticRegParams format."""
    p = _to_named_params(params)
    key1, key2 = random.split(key)

    new_alpha = p.alpha + 0.1 * random.normal(key1)
    new_betas = p.betas + 0.1 * random.normal(key2, shape=p.betas.shape)
    return LogisticRegParams(alpha=new_alpha, betas=new_betas)


In [68]:
from functools import partial
import jax
import jax.numpy as jnp
from jax import random


def gen_initial_samples(no_samples: int, p: int, key):
    """SMC particle matrix: [alpha, betas...] with prior draws."""
    k1, k2 = random.split(key)
    alphas = 3.0 * random.normal(k1, shape=(no_samples, 1))
    betas = 1.5 * random.normal(k2, shape=(no_samples, p))
    return jnp.concatenate([alphas, betas], axis=1)


# Convert pandas to jax arrays
X_jax = jnp.array(chosen_X.values, dtype=float)
y_jax = jnp.array(chosen_y.values, dtype=float).ravel()
p = X_jax.shape[1]


def _vector_to_named(vec):
    vec = jnp.ravel(vec)
    return LogisticRegParams(alpha=vec[0], betas=vec[1:])


def _target_logpdf_single(vec, X_observed, y_observed):
    return posterior_log_pdf(_vector_to_named(vec), X_observed, y_observed)


def _prior_logpdf_single(vec):
    return prior_log_pdf(_vector_to_named(vec))


def target_logpdf_vec(vec_or_batch, X_observed, y_observed):
    arr = jnp.asarray(vec_or_batch)
    if arr.ndim == 1:
        return _target_logpdf_single(arr, X_observed, y_observed)
    return jax.vmap(lambda row: _target_logpdf_single(row, X_observed, y_observed))(arr)


def prior_logpdf_vec(vec_or_batch):
    arr = jnp.asarray(vec_or_batch)
    if arr.ndim == 1:
        return _prior_logpdf_single(arr)
    return jax.vmap(_prior_logpdf_single)(arr)


def proposed_fn_vec(vec, key):
    out = proposed_fn(_vector_to_named(vec), key)
    return jnp.concatenate([jnp.array([out.alpha]), jnp.ravel(out.betas)])


smc = SMC(
    dims=1 + p,
    target_dist_logpdf=partial(target_logpdf_vec, X_observed=X_jax, y_observed=y_jax),
    prior_dist_logpdf=prior_logpdf_vec,
    proposed_fn=proposed_fn_vec,
    key=random.key(0),
)

no_samples = 10_000
key = random.key(42)
initial_samples = gen_initial_samples(no_samples, p, key)
smc.reset(samples=initial_samples)

lam_list, tot_log_z = smc.build_intermediate_dists()
print("lambda schedule:", lam_list)
print("log evidence estimate:", float(tot_log_z))


p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  JitTracer(float32[])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  JitTracer(float32[])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)
betas shape:  (8,)
p.alpha:  VmapTracer(aval=float32[], batched=float32[10000])
X_observed shape:  (270, 8)

In [93]:
import jax.numpy as jnp

# SMC estimate of normalizing constant (model evidence)
log_z_hat = jnp.asarray(tot_log_z)
# guard overflow when exponentiating large values
z_hat = jnp.exp(jnp.clip(log_z_hat, min=-700.0, max=700.0))

print(f"SMC estimate log(Z): {float(log_z_hat):.6f}")
print(f"SMC estimate Z: {float(z_hat):.6e}")

TypeError: clip() got an unexpected keyword argument 'a_max'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _extract_particles_array(smc_obj):
    """Return SMC particles as 2D numpy array: (n_particles, 1 + p)."""
    particles = np.array(smc_obj.get_current_sample_list())
    if particles.ndim != 2:
        raise ValueError(f"Expected 2D particle matrix, got shape {particles.shape}")
    if particles.shape[1] < 2:
        raise ValueError(f"Expected at least alpha + 1 beta, got shape {particles.shape}")
    return particles


def plot_smc_traces(smc_obj, max_beta_legend=10, figsize=(14, 9)):
    """Plot particle-index traces for alpha and betas."""
    particles = _extract_particles_array(smc_obj)
    alpha_trace = particles[:, 0]
    betas_trace = particles[:, 1:]

    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True)

    axes[0].plot(alpha_trace, lw=0.8, alpha=0.9)
    axes[0].set_title("SMC Particle Trace - alpha")
    axes[0].set_ylabel("alpha")
    axes[0].grid(alpha=0.25)

    for j in range(betas_trace.shape[1]):
        axes[1].plot(betas_trace[:, j], lw=0.7, alpha=0.7, label=f"beta_{j+1}")

    axes[1].set_title("SMC Particle Traces - betas")
    axes[1].set_xlabel("particle index")
    axes[1].set_ylabel("beta value")
    axes[1].grid(alpha=0.25)

    if betas_trace.shape[1] <= max_beta_legend:
        axes[1].legend(loc="best", ncol=2, fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"particles shape: {particles.shape}")


# Usage
plot_smc_traces(smc)


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


def plot_smc_histograms(smc_obj, bins=30, figsize_per_panel=(4.2, 3.2)):
    """Plot histograms for alpha and each beta from final SMC particles."""
    particles = _extract_particles_array(smc_obj)
    n_params = particles.shape[1]  # alpha + betas
    n_cols = min(3, n_params)
    n_rows = math.ceil(n_params / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1] * n_rows),
    )
    axes = np.atleast_1d(axes).reshape(n_rows, n_cols)

    labels = ["alpha"] + [f"beta_{j+1}" for j in range(n_params - 1)]

    for idx in range(n_rows * n_cols):
        r, c = divmod(idx, n_cols)
        ax = axes[r, c]
        if idx < n_params:
            ax.hist(particles[:, idx], bins=bins, alpha=0.8)
            ax.set_title(labels[idx])
            ax.grid(alpha=0.25)
        else:
            ax.axis("off")

    fig.suptitle("SMC Posterior Particle Histograms", y=1.02)
    plt.tight_layout()
    plt.show()


# Usage
plot_smc_histograms(smc)


### Stage 2: Evidence-based Variable Selection 

- Choose a small set of 8 predictors.
- Define M as all $2^8$ subsets.
- For each M, run SMC and est $\log{p_M(y)}$
- Rank models by est log evidence.
- Inspect top model and evidence gaps.
- Check SMC stability.
- Evaluate posterior perf of held-out data.

Repeat same thing with stage 1 

### Stage 3: Larger Model Space

- Introduce a model prior $p(M)$
- Try with different search strategies: Greedy, Forward-Backward, Stochastic.
- Compare selected one with Lasso Logistic Regression and BIC.
- Study prior sensitivity.

This is very useful for the case where there are lots of parameters options.